In [9]:
!pip install torchaudio torch scipy datasets soundfile

In [2]:
import torch
import torchaudio
import torch.nn.functional as F
import urllib.request
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchaudio Version: {torchaudio.__version__}")

PyTorch Version: 2.11.0+cpu
Torchaudio Version: 2.11.0+cpu


In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We load the "B1" model size fine-tuned with a large margin (lm) on VoxCeleb data.
print("Downloading ReDimNet2 model...")
model = torch.hub.load(
    "PalabraAI/redimnet2",
    "redimnet2",
    model_name="b1",
    train_type="lm",
    pretrained=True
)

model = model.to(device)
model.eval() # Set to evaluation mode (turns off dropout/batchnorm updates)
print("Model loaded successfully!")

Using device: cpu
The repository PalabraAI_redimnet2 does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/PalabraAI/redimnet2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/PalabraAI/redimnet2/releases/download/v1.0.0/b1-vox2-lm.pt" to /root/.cache/torch/hub/checkpoints/b1-vox2-lm.pt


100%|██████████| 9.36M/9.36M [00:00<00:00, 49.1MB/s]


Model loaded successfully!


In [4]:
def extract_embedding(audio_path, model, device):
    """Loads 16kHz audio and extracts a normalized ReDimNet2 embedding."""
    # 1. Load audio
    waveform, sample_rate = torchaudio.load(audio_path)

    # 2. Resample if the audio is not 16 kHz (ReDimNet expects 16kHz)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    # 3. Convert stereo to mono if necessary
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    waveform = waveform.to(device)

    # 4. Extract embedding (No gradient calculation needed for inference)
    with torch.no_grad():
        embedding = model(waveform)

    # ReDimNet outputs L2-normalized embeddings, but we enforce it here just to be safe
    return F.normalize(embedding, p=2, dim=1)

def verify_voice(emb1, emb2, threshold=0.45):
    """Compares two embeddings using Cosine Similarity."""
    # Cosine similarity ranges from -1 to 1. Higher means more similar.
    similarity = F.cosine_similarity(emb1, emb2).item()

    print(f"Similarity Score: {similarity:.4f} (Threshold: {threshold})")
    if similarity > threshold:
        print("✅ AUTHENTICATION SUCCESSFUL: Voices match.")
    else:
        print("❌ AUTHENTICATION FAILED: Voices do not match.")

    return similarity

In [10]:
from datasets import load_dataset
import soundfile as sf

print("Loading LibriSpeech samples...")

dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="train.100",
    streaming=True
)

speaker_A = []
speaker_B = None

for example in dataset:

    # Find two recordings from the same speaker
    if len(speaker_A) == 0:
        speaker_A.append(example)

    elif example["speaker_id"] == speaker_A[0]["speaker_id"]:
        speaker_A.append(example)

    # Find a different speaker
    elif speaker_B is None:
        speaker_B = example

    if len(speaker_A) >= 2 and speaker_B is not None:
        break


# Save audio files
files = {
    "speaker_A_enroll.wav": speaker_A[0],
    "speaker_A_test.wav": speaker_A[1],
    "speaker_B_test.wav": speaker_B,
}

for filename, example in files.items():

    audio = example["audio"]

    sf.write(
        filename,
        audio["array"],
        audio["sampling_rate"]
    )

    print(
        f"Saved {filename} | "
        f"speaker_id={example['speaker_id']} | "
        f"id={example['id']}"
    )


# Extract Embeddings
print("\nExtracting voice biometrics...")

emb_A_enroll = extract_embedding(
    "speaker_A_enroll.wav",
    model,
    device
)

emb_A_test = extract_embedding(
    "speaker_A_test.wav",
    model,
    device
)

emb_B_test = extract_embedding(
    "speaker_B_test.wav",
    model,
    device
)


# Perform Biometric Verification
print("\n--- TEST 1: Same Speaker (Should Succeed) ---")

verify_voice(
    emb_A_enroll,
    emb_A_test,
    threshold=0.45
)


print("\n--- TEST 2: Imposter Speaker (Should Fail) ---")

verify_voice(
    emb_A_enroll,
    emb_B_test,
    threshold=0.45
)

Loading LibriSpeech samples...


README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Saved speaker_A_enroll.wav | speaker_id=374 | id=374-180298-0000
Saved speaker_A_test.wav | speaker_id=374 | id=374-180298-0001
Saved speaker_B_test.wav | speaker_id=7800 | id=7800-283478-0000

Extracting voice biometrics...

--- TEST 1: Same Speaker (Should Succeed) ---
Similarity Score: 0.9636 (Threshold: 0.45)
✅ AUTHENTICATION SUCCESSFUL: Voices match.

--- TEST 2: Imposter Speaker (Should Fail) ---
Similarity Score: 0.0809 (Threshold: 0.45)
❌ AUTHENTICATION FAILED: Voices do not match.


0.08089542388916016